Alpha Vantage API → Python → Pandas → SQL Server


In [37]:
import pandas as pd
import requests
from dotenv import load_dotenv
import os

load_dotenv()

API_KEY = os.getenv("API_KEY")
SYMBOL = "AAPL"
BASE_URL = "https://www.alphavantage.co/query"

params = {
    "function": "TIME_SERIES_DAILY",
    "symbol": SYMBOL,
    "outputsize": "compact",
    "apikey": API_KEY
}

response = requests.get(BASE_URL,params=params)
data = response.json()

print(data.keys())

dict_keys(['Meta Data', 'Time Series (Daily)'])


In [38]:
print(data['Meta Data'])

{'1. Information': 'Daily Prices (open, high, low, close) and Volumes', '2. Symbol': 'AAPL', '3. Last Refreshed': '2026-03-16', '4. Output Size': 'Compact', '5. Time Zone': 'US/Eastern'}


In [39]:
time_series = data['Time Series (Daily)']

#Ver la primera fecha disponible
first_key = list(time_series.keys())[0]
print(first_key)
print(time_series[first_key])

2026-03-16
{'1. open': '252.1050', '2. high': '253.8850', '3. low': '249.8800', '4. close': '252.8200', '5. volume': '32074209'}


In [40]:
#convertir todo ese diccionario en un DataFrame de Pandas.
df = pd.DataFrame.from_dict(time_series,orient="index")

print(df.shape)
print(df.head())

(100, 5)
             1. open   2. high    3. low  4. close 5. volume
2026-03-16  252.1050  253.8850  249.8800  252.8200  32074209
2026-03-13  255.4800  256.3300  249.5200  250.1200  36929988
2026-03-12  258.6600  258.9500  254.1800  255.7600  40794020
2026-03-11  261.0900  262.1300  259.5500  260.8100  26218927
2026-03-10  257.6450  262.4800  256.9500  260.8300  30590765


In [41]:
df.columns = ["open","high","low","close","volume"]
df.index = pd.to_datetime(df.index)
df = df.astype(float)

print(df.dtypes)
print(df.head())

open      float64
high      float64
low       float64
close     float64
volume    float64
dtype: object
               open     high     low   close      volume
2026-03-16  252.105  253.885  249.88  252.82  32074209.0
2026-03-13  255.480  256.330  249.52  250.12  36929988.0
2026-03-12  258.660  258.950  254.18  255.76  40794020.0
2026-03-11  261.090  262.130  259.55  260.81  26218927.0
2026-03-10  257.645  262.480  256.95  260.83  30590765.0


In [42]:
#Orden ascendente
df = df.sort_index()
print(df.head())

              open    high       low   close      volume
2025-10-21  261.88  265.29  261.8300  262.77  46695948.0
2025-10-22  262.65  262.85  255.4300  258.45  45015254.0
2025-10-23  259.94  260.62  258.0101  259.58  32754941.0
2025-10-24  261.19  264.13  259.1800  262.82  38253717.0
2025-10-27  264.88  269.12  264.6501  268.81  44888152.0


In [43]:
#columna con symbolo de la acción
df["symbol"] = SYMBOL
print(df.head())

              open    high       low   close      volume symbol
2025-10-21  261.88  265.29  261.8300  262.77  46695948.0   AAPL
2025-10-22  262.65  262.85  255.4300  258.45  45015254.0   AAPL
2025-10-23  259.94  260.62  258.0101  259.58  32754941.0   AAPL
2025-10-24  261.19  264.13  259.1800  262.82  38253717.0   AAPL
2025-10-27  264.88  269.12  264.6501  268.81  44888152.0   AAPL


In [44]:
# La conexión está configurada correctamente.
from sqlalchemy import create_engine

engine = create_engine(
    "mssql+pyodbc://localhost\\SQLEXPRESS/financial_analysis"
    "?driver=ODBC+Driver+17+for+SQL+Server"
    "&trusted_connection=yes")
print(engine)

Engine(mssql+pyodbc://localhost\SQLEXPRESS/financial_analysis?driver=ODBC+Driver+17+for+SQL+Server&trusted_connection=yes)


In [45]:
df.to_sql(
    name="stock_prices",
    con=engine,
    if_exists="replace",
    index=True,
    index_label="date"
)
print("Carga exitosa")

Carga exitosa


c:\Users\leoje\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\pandas\io\sql.py:1649: SAWarning: Unrecognized server version info '17.0.1105.2'.  Some SQL Server features may not function properly.
  con = self.exit_stack.enter_context(con.connect())


In [46]:
import subprocess
subprocess.run(["pip","install","python-dotenv"],check=True)

FileNotFoundError: [WinError 2] El sistema no puede encontrar el archivo especificado